# Day 2 — Silver and Quality Engineering

This notebook is adapted to the current Healthcare dataset, Catalog, Bronze tables and Volume structure. It creates curated Silver tables, a DQ rule catalogue, and a central rejected-records quarantine table.

## 1. Imports and table configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CATALOG = "capstone_8hospital"
BRONZE_SCHEMA = "bronze_schema"
SILVER_SCHEMA = "silver_schema"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

BRONZE_ENCOUNTER_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_patient_encounters"
BRONZE_LAB_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_lab_test_events"
BRONZE_TRANSFER_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_bed_transfer_events"
BRONZE_PATIENT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_patients"
BRONZE_DEPARTMENT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_departments"
BRONZE_BED_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_beds"

SILVER_PATIENT_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_patients"
SILVER_DEPARTMENT_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_departments"
SILVER_BED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_beds"
SILVER_ENCOUNTER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_patient_encounters"
SILVER_LAB_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_lab_test_events"
SILVER_TRANSFER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_bed_transfer_events"
SILVER_REJECTED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_rejected_records"
SILVER_DQ_RULE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_dq_rule_catalogue"
SILVER_DUPLICATE_AUDIT_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_duplicate_audit"

## 2. Create the Data Quality Rules Catalogue table

Rules are based on the columns present in the current Healthcare source files. Missing optional diagnosis and clinician values are retained and flagged, not rejected. Duplicate business-key extras are recorded in the central quarantine table with a rule ID.

In [0]:
rule_rows = [
    ("ENC-01", "encounter_id", "Required encounter identifier is present", "ERROR", "REJECT"),
    ("ENC-02", "admission_ts", "Admission timestamp is valid", "ERROR", "REJECT"),
    ("ENC-03", "patient_id", "Patient key exists in patient reference", "ERROR", "REJECT"),
    ("ENC-04", "department_id", "Department key exists in department reference", "ERROR", "REJECT"),
    ("ENC-05", "bed_id", "Bed key exists when supplied", "ERROR", "REJECT"),
    ("ENC-06", "discharge_ts", "Discharge timestamp is not before admission", "ERROR", "REJECT"),
    ("ENC-07", "encounter_id", "Only one encounter row is retained per business key", "WARNING", "QUARANTINE_EXTRA"),
    ("ENC-08", "diagnosis_group, attending_clinician_id", "Optional values are retained and flagged when missing", "WARNING", "FLAG"),
    ("LAB-01", "lab_event_id", "Required lab event identifier is present", "ERROR", "REJECT"),
    ("LAB-02", "ordered_ts", "Lab order timestamp is valid", "ERROR", "REJECT"),
    ("LAB-03", "encounter_id", "Encounter key exists in Silver encounters", "ERROR", "REJECT"),
    ("LAB-04", "collected_ts, result_ts", "Collection and result timestamps follow order chronology when present", "ERROR", "REJECT"),
    ("LAB-05", "lab_event_id", "Only one lab row is retained per business key", "WARNING", "QUARANTINE_EXTRA"),
    ("TRF-01", "transfer_id", "Required transfer identifier is present", "ERROR", "REJECT"),
    ("TRF-02", "transfer_ts", "Transfer timestamp is valid", "ERROR", "REJECT"),
    ("TRF-03", "encounter_id", "Encounter key exists in Silver encounters", "ERROR", "REJECT"),
    ("TRF-04", "from_bed_id, to_bed_id", "Supplied bed keys exist in bed reference; unmatched keys are retained and flagged", "WARNING", "FLAG"),
    ("TRF-05", "transfer_id", "Only one transfer row is retained per business key", "WARNING", "QUARANTINE_EXTRA"),
    ("REF-01", "patient_id, department_id, bed_id", "Master business key is present", "ERROR", "REJECT"),
    ("REF-02", "patient_id, department_id, bed_id", "Master business keys are unique", "WARNING", "QUARANTINE_EXTRA")
]

rule_schema = T.StructType([
    T.StructField("rule_id", T.StringType(), False),
    T.StructField("field_name", T.StringType(), False),
    T.StructField("rule_description", T.StringType(), False),
    T.StructField("severity", T.StringType(), False),
    T.StructField("action", T.StringType(), False)
])

dq_rules = spark.createDataFrame(rule_rows, schema=rule_schema).withColumn(
    "updated_at", F.current_timestamp()
)

(dq_rules.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(SILVER_DQ_RULE_TABLE))

display(dq_rules.orderBy("rule_id"))
print("Saved:", SILVER_DQ_RULE_TABLE)

rule_id,field_name,rule_description,severity,action,updated_at
ENC-01,encounter_id,Required encounter identifier is present,ERROR,REJECT,2026-11-04T08:38:03.518Z
ENC-02,admission_ts,Admission timestamp is valid,ERROR,REJECT,2026-11-04T08:38:03.518Z
ENC-03,patient_id,Patient key exists in patient reference,ERROR,REJECT,2026-11-04T08:38:03.518Z
ENC-04,department_id,Department key exists in department reference,ERROR,REJECT,2026-11-04T08:38:03.518Z
ENC-05,bed_id,Bed key exists when supplied,ERROR,REJECT,2026-11-04T08:38:03.518Z
ENC-06,discharge_ts,Discharge timestamp is not before admission,ERROR,REJECT,2026-11-04T08:38:03.518Z
ENC-07,encounter_id,Only one encounter row is retained per business key,WARNING,QUARANTINE_EXTRA,2026-11-04T08:38:03.518Z
ENC-08,"diagnosis_group, attending_clinician_id",Optional values are retained and flagged when missing,WARNING,FLAG,2026-11-04T08:38:03.518Z
LAB-01,lab_event_id,Required lab event identifier is present,ERROR,REJECT,2026-11-04T08:38:03.518Z
LAB-02,ordered_ts,Lab order timestamp is valid,ERROR,REJECT,2026-11-04T08:38:03.518Z


Saved: capstone_8hospital.silver_schema.silver_dq_rule_catalogue


## 3. Read Bronze tables and inspect schemas

In [0]:
encounters = spark.table(BRONZE_ENCOUNTER_TABLE)
labs = spark.table(BRONZE_LAB_TABLE)
transfers = spark.table(BRONZE_TRANSFER_TABLE)
patients = spark.table(BRONZE_PATIENT_TABLE)
departments = spark.table(BRONZE_DEPARTMENT_TABLE)
beds = spark.table(BRONZE_BED_TABLE)

for name, dataframe in [
    ("Bronze patient encounters", encounters),
    ("Bronze lab test events", labs),
    ("Bronze bed transfer events", transfers),
    ("Bronze patients", patients),
    ("Bronze departments", departments),
    ("Bronze beds", beds)
]:
    print(name, dataframe.count())
    dataframe.printSchema()

Bronze patient encounters 12376
root
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- admission_ts: string (nullable = true)
 |-- discharge_ts: string (nullable = true)
 |-- encounter_type: string (nullable = true)
 |-- triage_priority: string (nullable = true)
 |-- encounter_status: string (nullable = true)
 |-- attending_clinician_id: string (nullable = true)
 |-- admission_source: string (nullable = true)
 |-- diagnosis_group: string (nullable = true)
 |-- bed_id: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

Bronze lab test events 758
root
 |-- lab_event_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- test_code: string (nullable = true)
 |-- ordered_ts: string (nullable = true)
 |-- collected_ts: string (n

## 4. Clean and validate master reference tables

In [0]:
def add_lineage_columns(dataframe):
    if "source_file" not in dataframe.columns:
        dataframe = dataframe.withColumn("source_file", F.lit(None).cast("string"))
    if "ingestion_timestamp" not in dataframe.columns:
        dataframe = dataframe.withColumn("ingestion_timestamp", F.lit(None).cast("timestamp"))
    return dataframe

patients = add_lineage_columns(patients)
departments = add_lineage_columns(departments)
beds = add_lineage_columns(beds)

patients_typed = (
    patients
    .withColumn("patient_id", F.trim(F.col("patient_id").cast("string")))
    .withColumn("synthetic_patient_code", F.trim(F.col("synthetic_patient_code").cast("string")))
    .withColumn("sex", F.upper(F.trim(F.col("sex").cast("string"))))
    .withColumn("city", F.trim(F.col("city").cast("string")))
    .withColumn("registration_date", F.to_date("registration_date"))
    .withColumn("consent_flag", F.upper(F.trim(F.col("consent_flag").cast("string"))))
    .withColumn("insurance_type", F.upper(F.trim(F.col("insurance_type").cast("string"))))
    .withColumn("patient_status", F.upper(F.trim(F.col("patient_status").cast("string"))))
    .withColumn("primary_language", F.trim(F.col("primary_language").cast("string")))
)

departments_typed = (
    departments
    .withColumn("department_id", F.trim(F.col("department_id").cast("string")))
    .withColumn("department_name", F.trim(F.col("department_name").cast("string")))
    .withColumn("specialty", F.trim(F.col("specialty").cast("string")))
    .withColumn("hospital_site", F.trim(F.col("hospital_site").cast("string")))
    .withColumn("floor", F.col("floor").cast("int"))
    .withColumn("service_level", F.upper(F.trim(F.col("service_level").cast("string"))))
    .withColumn("active_flag", F.upper(F.trim(F.col("active_flag").cast("string"))))
    .withColumn("head_clinician_id", F.trim(F.col("head_clinician_id").cast("string")))
    .withColumn("capacity_beds", F.col("capacity_beds").cast("int"))
    .withColumn("region", F.trim(F.col("region").cast("string")))
)

beds_typed = (
    beds
    .withColumn("bed_id", F.trim(F.col("bed_id").cast("string")))
    .withColumn("department_id", F.trim(F.col("department_id").cast("string")))
    .withColumn("ward", F.trim(F.col("ward").cast("string")))
    .withColumn("bed_type", F.upper(F.trim(F.col("bed_type").cast("string"))))
    .withColumn("isolation_capable", F.upper(F.trim(F.col("isolation_capable").cast("string"))))
    .withColumn("bed_status", F.upper(F.trim(F.col("bed_status").cast("string"))))
    .withColumn("floor", F.col("floor").cast("int"))
    .withColumn("hospital_site", F.trim(F.col("hospital_site").cast("string")))
    .withColumn("active_flag", F.upper(F.trim(F.col("active_flag").cast("string"))))
    .withColumn("effective_date", F.to_date("effective_date"))
)

## 5. Apply master-key validation and deterministic deduplication

In [0]:
def prepare_master(dataframe, key, source_entity):
    checked = (
        dataframe
        .withColumn(
            "rule_id",
            F.when(F.col(key).isNull() | (F.trim(F.col(key)) == ""), "REF-01")
        )
        .withColumn(
            "rejection_reason",
            F.when(F.col("rule_id") == "REF-01", F.lit(f"Missing or blank {key}"))
        )
        .withColumn("source_entity", F.lit(source_entity))
        .withColumn("rejected_at", F.current_timestamp())
    )

    invalid = checked.filter(F.col("rule_id").isNotNull())
    valid = checked.filter(F.col("rule_id").isNull())

    order_cols = [F.col("ingestion_timestamp").desc_nulls_last(),
                  F.col("source_file").asc_nulls_last()]
    valid = valid.withColumn("_row_hash", F.sha2(F.to_json(F.struct(*[F.col(c) for c in dataframe.columns])), 256))
    order_cols.append(F.col("_row_hash").asc())

    ranked = valid.withColumn(
        "_row_num",
        F.row_number().over(Window.partitionBy(key).orderBy(*order_cols))
    )

    duplicates = (
        ranked.filter(F.col("_row_num") > 1)
        .withColumn("rule_id", F.lit("REF-02"))
        .withColumn("rejection_reason", F.lit(f"Duplicate {key}; deterministic survivor retained"))
        .withColumn("rejected_at", F.current_timestamp())
    )

    clean = ranked.filter(F.col("_row_num") == 1).drop("_row_num", "_row_hash", "rule_id", "rejection_reason", "source_entity", "rejected_at")
    rejected = invalid.unionByName(duplicates, allowMissingColumns=True)
    return clean, rejected, duplicates

silver_patients, rejected_patients, duplicate_patients = prepare_master(patients_typed, "patient_id", "PATIENTS")
silver_departments, rejected_departments, duplicate_departments = prepare_master(departments_typed, "department_id", "DEPARTMENTS")
silver_beds, rejected_beds, duplicate_beds = prepare_master(beds_typed, "bed_id", "BEDS")

display(spark.createDataFrame([
    ("Patients", patients.count(), silver_patients.count(), rejected_patients.count()),
    ("Departments", departments.count(), silver_departments.count(), rejected_departments.count()),
    ("Beds", beds.count(), silver_beds.count(), rejected_beds.count())
], ["source", "bronze_rows", "silver_rows", "quarantine_rows"]))

source,bronze_rows,silver_rows,quarantine_rows
Patients,183,183,0
Departments,192,192,0
Beds,201,201,0


## 6. Cast and standardize patient encounter data

In [0]:
encounters = add_lineage_columns(encounters)
encounter_source_columns = encounters.columns

encounters_typed = encounters
for column_name in ["admission_ts", "discharge_ts"]:
    encounters_typed = encounters_typed.withColumn(
        f"{column_name}_raw", F.col(column_name).cast("string")
    )

encounters_typed = encounters_typed.withColumn(
    "_source_row_hash",
    F.sha2(F.to_json(F.struct(*[F.col(c) for c in encounter_source_columns])), 256)
)

encounters_typed = (
    encounters_typed
    .withColumn("encounter_id", F.trim(F.col("encounter_id").cast("string")))
    .withColumn("patient_id", F.trim(F.col("patient_id").cast("string")))
    .withColumn("department_id", F.trim(F.col("department_id").cast("string")))
    .withColumn("bed_id", F.trim(F.col("bed_id").cast("string")))
    .withColumn("admission_ts", F.to_timestamp("admission_ts"))
    .withColumn("discharge_ts", F.to_timestamp("discharge_ts"))
    .withColumn("encounter_type", F.upper(F.trim(F.col("encounter_type").cast("string"))))
    .withColumn("triage_priority", F.upper(F.trim(F.col("triage_priority").cast("string"))))
    .withColumn("encounter_status", F.upper(F.trim(F.col("encounter_status").cast("string"))))
    .withColumn("attending_clinician_id", F.trim(F.col("attending_clinician_id").cast("string")))
    .withColumn("admission_source", F.upper(F.trim(F.col("admission_source").cast("string"))))
    .withColumn("diagnosis_group", F.upper(F.trim(F.col("diagnosis_group").cast("string"))))
)

display(encounters_typed.select(
    "encounter_id", "admission_ts_raw", "admission_ts",
    "discharge_ts_raw", "discharge_ts", "encounter_status"
).limit(10))

encounter_id,admission_ts_raw,admission_ts,discharge_ts_raw,discharge_ts,encounter_status
ENCOUNT080003715,2026-07-19T05:48:00Z,2026-07-19T05:48:00.000Z,2026-07-19T05:53:00Z,2026-07-19T05:53:00.000Z,ADMITTED
ENCOUNT080003716,2026-07-19T05:55:00Z,2026-07-19T05:55:00.000Z,2026-07-19T06:00:00Z,2026-07-19T06:00:00.000Z,TRANSFERRED
ENCOUNT080003717,2026-07-19T06:02:00Z,2026-07-19T06:02:00.000Z,2026-07-19T06:07:00Z,2026-07-19T06:07:00.000Z,DISCHARGED
ENCOUNT080003718,2026-07-19T06:09:00Z,2026-07-19T06:09:00.000Z,2026-07-19T06:14:00Z,2026-07-19T06:14:00.000Z,TRANSFERRED
ENCOUNT080003719,2026-07-19T06:16:00Z,2026-07-19T06:16:00.000Z,2026-07-19T06:21:00Z,2026-07-19T06:21:00.000Z,TRANSFERRED
ENCOUNT080003720,2026-07-19T06:23:00Z,2026-07-19T06:23:00.000Z,2026-07-19T06:28:00Z,2026-07-19T06:28:00.000Z,TRANSFERRED
ENCOUNT080003721,2026-07-19T06:30:00Z,2026-07-19T06:30:00.000Z,2026-07-19T06:35:00Z,2026-07-19T06:35:00.000Z,ADMITTED
ENCOUNT080003722,2026-07-19T06:37:00Z,2026-07-19T06:37:00.000Z,2026-07-19T06:42:00Z,2026-07-19T06:42:00.000Z,TRANSFERRED
ENCOUNT080003723,2026-07-19T06:44:00Z,2026-07-19T06:44:00.000Z,2026-07-19T06:49:00Z,2026-07-19T06:49:00.000Z,DISCHARGED
ENCOUNT080003724,2026-07-19T06:51:00Z,2026-07-19T06:51:00.000Z,2026-07-19T06:56:00Z,2026-07-19T06:56:00.000Z,DISCHARGED


## 7. Apply encounter DQ rules and reference checks

In [0]:
patient_keys = silver_patients.select("patient_id").dropDuplicates()
department_keys = silver_departments.select("department_id").dropDuplicates()
bed_keys = silver_beds.select("bed_id").dropDuplicates()

encounters_checked = (
    encounters_typed
    .withColumn(
        "rule_id",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), "ENC-01")
         .when(F.col("admission_ts").isNull(), "ENC-02")
         .when(F.col("discharge_ts").isNotNull() & (F.col("discharge_ts") < F.col("admission_ts")), "ENC-06")
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rule_id") == "ENC-01", "Missing or blank encounter_id")
         .when(F.col("rule_id") == "ENC-02", "Missing or invalid admission_ts; raw value retained")
         .when(F.col("rule_id") == "ENC-06", "discharge_ts is earlier than admission_ts")
    )
    .withColumn("source_entity", F.lit("PATIENT_ENCOUNTERS"))
    .withColumn("rejected_at", F.current_timestamp())
)

encounters_with_refs = (
    encounters_checked
    .join(patient_keys.withColumn("_patient_found", F.lit(True)), "patient_id", "left")
    .join(department_keys.withColumn("_department_found", F.lit(True)), "department_id", "left")
    .join(bed_keys.withColumn("_bed_found", F.lit(True)), "bed_id", "left")
    .withColumn(
        "rule_id",
        F.when(F.col("rule_id").isNotNull(), F.col("rule_id"))
         .when(F.col("_patient_found").isNull(), "ENC-03")
         .when(F.col("_department_found").isNull(), "ENC-04")
         .when(F.col("bed_id").isNotNull() & F.col("_bed_found").isNull(), "ENC-05")
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rejection_reason").isNotNull(), F.col("rejection_reason"))
         .when(F.col("rule_id") == "ENC-03", "patient_id not found in Silver patient reference")
         .when(F.col("rule_id") == "ENC-04", "department_id not found in Silver department reference")
         .when(F.col("rule_id") == "ENC-05", "bed_id not found in Silver bed reference")
    )
)

encounter_rejected_basic = encounters_with_refs.filter(F.col("rule_id").isNotNull())
encounter_valid_basic = encounters_with_refs.filter(F.col("rule_id").isNull())

print("Encounter rows:", encounters.count())
print("Rejected before deduplication:", encounter_rejected_basic.count())
print("Valid before deduplication:", encounter_valid_basic.count())

Encounter rows: 12376
Rejected before deduplication: 0
Valid before deduplication: 12376


## 8. Deduplicate encounters and calculate valid stay intervals

In [0]:
encounter_order = [
    F.col("ingestion_timestamp").desc_nulls_last(),
    F.col("source_file").asc_nulls_last(),
    F.col("_source_row_hash").asc()
]

encounter_ranked = encounter_valid_basic.withColumn(
    "_row_num",
    F.row_number().over(Window.partitionBy("encounter_id").orderBy(*encounter_order))
)

encounter_duplicate_rows = (
    encounter_ranked.filter(F.col("_row_num") > 1)
    .withColumn("rule_id", F.lit("ENC-07"))
    .withColumn("rejection_reason", F.lit("Duplicate encounter_id; deterministic survivor retained"))
    .withColumn("source_entity", F.lit("PATIENT_ENCOUNTERS"))
    .withColumn("rejected_at", F.current_timestamp())
)

encounter_survivors = encounter_ranked.filter(F.col("_row_num") == 1)

optional_missing = (
    F.col("diagnosis_group").isNull() | (F.trim(F.col("diagnosis_group")) == "") |
    F.col("attending_clinician_id").isNull() | (F.trim(F.col("attending_clinician_id")) == "")
)

silver_encounters = (
    encounter_survivors
    .withColumn("missing_optional_category_flag", optional_missing)
    .withColumn("dq_status", F.when(optional_missing, "FLAGGED").otherwise("VALID"))
    .withColumn(
        "length_of_stay_minutes",
        F.when(
            F.col("discharge_ts").isNotNull(),
            (F.col("discharge_ts").cast("long") - F.col("admission_ts").cast("long")) / 60
        ).cast("double")
    )
    .drop("_row_num", "_patient_found", "_department_found", "_bed_found",
          "rule_id", "rejection_reason", "source_entity", "rejected_at", "_source_row_hash")
)

display(silver_encounters.groupBy("dq_status", "missing_optional_category_flag").count())
print("Silver encounters:", silver_encounters.count())
print("Duplicate encounter rows quarantined:", encounter_duplicate_rows.count())

dq_status,missing_optional_category_flag,count
VALID,false,12317
FLAGGED,true,12


Silver encounters: 12329
Duplicate encounter rows quarantined: 47


## 9. Clean and validate lab test events

In [0]:
labs = add_lineage_columns(labs)
lab_source_columns = labs.columns

labs_typed = labs
for column_name in ["ordered_ts", "collected_ts", "result_ts"]:
    labs_typed = labs_typed.withColumn(f"{column_name}_raw", F.col(column_name).cast("string"))

labs_typed = labs_typed.withColumn(
    "_source_row_hash",
    F.sha2(F.to_json(F.struct(*[F.col(c) for c in lab_source_columns])), 256)
)

labs_typed = (
    labs_typed
    .withColumn("lab_event_id", F.trim(F.col("lab_event_id").cast("string")))
    .withColumn("encounter_id", F.trim(F.col("encounter_id").cast("string")))
    .withColumn("patient_id", F.trim(F.col("patient_id").cast("string")))
    .withColumn("test_code", F.upper(F.trim(F.col("test_code").cast("string"))))
    .withColumn("ordered_ts", F.to_timestamp("ordered_ts"))
    .withColumn("collected_ts", F.to_timestamp("collected_ts"))
    .withColumn("result_ts", F.to_timestamp("result_ts"))
    .withColumn("result_status", F.upper(F.trim(F.col("result_status").cast("string"))))
    .withColumn("department_id", F.trim(F.col("department_id").cast("string")))
    .withColumn("priority", F.upper(F.trim(F.col("priority").cast("string"))))
)

encounter_keys = silver_encounters.select("encounter_id").dropDuplicates()

labs_checked = (
    labs_typed
    .withColumn(
        "rule_id",
        F.when(F.col("lab_event_id").isNull() | (F.col("lab_event_id") == ""), "LAB-01")
         .when(F.col("ordered_ts").isNull(), "LAB-02")
         .when(F.col("collected_ts").isNotNull() & (F.col("collected_ts") < F.col("ordered_ts")), "LAB-04")
         .when(F.col("result_ts").isNotNull() & F.col("collected_ts").isNotNull() & (F.col("result_ts") < F.col("collected_ts")), "LAB-04")
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rule_id") == "LAB-01", "Missing or blank lab_event_id")
         .when(F.col("rule_id") == "LAB-02", "Missing or invalid ordered_ts")
         .when(F.col("rule_id") == "LAB-04", "Lab timestamps are not in chronological order")
    )
    .withColumn("source_entity", F.lit("LAB_TEST_EVENTS"))
    .withColumn("rejected_at", F.current_timestamp())
)

labs_with_refs = (
    labs_checked
    .join(encounter_keys.withColumn("_encounter_found", F.lit(True)), "encounter_id", "left")
    .withColumn(
        "rule_id",
        F.when(F.col("rule_id").isNotNull(), F.col("rule_id"))
         .when(F.col("_encounter_found").isNull(), "LAB-03")
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rejection_reason").isNotNull(), F.col("rejection_reason"))
         .when(F.col("rule_id") == "LAB-03", "encounter_id not found in Silver encounters")
    )
)

lab_rejected_basic = labs_with_refs.filter(F.col("rule_id").isNotNull())
lab_valid_basic = labs_with_refs.filter(F.col("rule_id").isNull())

lab_ranked = lab_valid_basic.withColumn(
    "_row_num",
    F.row_number().over(Window.partitionBy("lab_event_id").orderBy(
        F.col("ingestion_timestamp").desc_nulls_last(),
        F.col("source_file").asc_nulls_last(),
        F.col("_source_row_hash").asc()
    ))
)

lab_duplicate_rows = (
    lab_ranked.filter(F.col("_row_num") > 1)
    .withColumn("rule_id", F.lit("LAB-05"))
    .withColumn("rejection_reason", F.lit("Duplicate lab_event_id; deterministic survivor retained"))
    .withColumn("source_entity", F.lit("LAB_TEST_EVENTS"))
    .withColumn("rejected_at", F.current_timestamp())
)

silver_labs = lab_ranked.filter(F.col("_row_num") == 1).drop(
    "_row_num", "_encounter_found", "_source_row_hash",
    "rule_id", "rejection_reason", "source_entity", "rejected_at"
)

print("Bronze lab events:", labs.count())
print("Silver lab events:", silver_labs.count())
print("Rejected lab events:", lab_rejected_basic.count())
print("Duplicate lab events:", lab_duplicate_rows.count())

Bronze lab events: 758
Silver lab events: 758
Rejected lab events: 0
Duplicate lab events: 0


## 10. Clean and validate bed transfer events

In [0]:
transfers = add_lineage_columns(transfers)
transfer_source_columns = transfers.columns

transfers_typed = transfers
for column_name in ["transfer_ts", "recorded_ts"]:
    transfers_typed = transfers_typed.withColumn(f"{column_name}_raw", F.col(column_name).cast("string"))

transfers_typed = transfers_typed.withColumn(
    "_source_row_hash",
    F.sha2(F.to_json(F.struct(*[F.col(c) for c in transfer_source_columns])), 256)
)

transfers_typed = (
    transfers_typed
    .withColumn("transfer_id", F.trim(F.col("transfer_id").cast("string")))
    .withColumn("encounter_id", F.trim(F.col("encounter_id").cast("string")))
    .withColumn("patient_id", F.trim(F.col("patient_id").cast("string")))
    .withColumn("from_bed_id", F.trim(F.col("from_bed_id").cast("string")))
    .withColumn("to_bed_id", F.trim(F.col("to_bed_id").cast("string")))
    .withColumn("transfer_ts", F.to_timestamp("transfer_ts"))
    .withColumn("transfer_reason", F.upper(F.trim(F.col("transfer_reason").cast("string"))))
    .withColumn("department_id", F.trim(F.col("department_id").cast("string")))
    .withColumn("transfer_status", F.upper(F.trim(F.col("transfer_status").cast("string"))))
    .withColumn("recorded_ts", F.to_timestamp("recorded_ts"))
)

transfer_bed_keys = silver_beds.select("bed_id").dropDuplicates()

transfers_checked = (
    transfers_typed
    .withColumn(
        "rule_id",
        F.when(F.col("transfer_id").isNull() | (F.col("transfer_id") == ""), "TRF-01")
         .when(F.col("transfer_ts").isNull(), "TRF-02")
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rule_id") == "TRF-01", "Missing or blank transfer_id")
         .when(F.col("rule_id") == "TRF-02", "Missing or invalid transfer_ts")
    )
    .withColumn("source_entity", F.lit("BED_TRANSFER_EVENTS"))
    .withColumn("rejected_at", F.current_timestamp())
)

transfers_with_refs = (
    transfers_checked
    .join(encounter_keys.withColumn("_encounter_found", F.lit(True)), "encounter_id", "left")
    .join(transfer_bed_keys.withColumnRenamed("bed_id", "_from_bed_key").withColumn("_from_bed_found", F.lit(True)),
          F.col("from_bed_id") == F.col("_from_bed_key"), "left")
    .join(transfer_bed_keys.withColumnRenamed("bed_id", "_to_bed_key").withColumn("_to_bed_found", F.lit(True)),
          F.col("to_bed_id") == F.col("_to_bed_key"), "left")
    .withColumn(
        "rule_id",
        F.when(F.col("rule_id").isNotNull(), F.col("rule_id"))
         .when(F.col("_encounter_found").isNull(), "TRF-03")
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rejection_reason").isNotNull(), F.col("rejection_reason"))
         .when(F.col("rule_id") == "TRF-03", "encounter_id not found in Silver encounters")
    )
    # TRF-04: source transfer bed IDs (FROM_BE..., TO_BED...) do not follow the bed master format (BED...),
    # so unmatched bed keys are retained and flagged instead of rejected
    .withColumn(
        "bed_reference_missing_flag",
        (F.col("from_bed_id").isNotNull() & F.col("_from_bed_found").isNull()) |
        (F.col("to_bed_id").isNotNull() & F.col("_to_bed_found").isNull())
    )
    .withColumn("dq_status", F.when(F.col("bed_reference_missing_flag"), "FLAGGED").otherwise("VALID"))
)

transfer_rejected_basic = transfers_with_refs.filter(F.col("rule_id").isNotNull())
transfer_valid_basic = transfers_with_refs.filter(F.col("rule_id").isNull())

transfer_ranked = transfer_valid_basic.withColumn(
    "_row_num",
    F.row_number().over(Window.partitionBy("transfer_id").orderBy(
        F.col("ingestion_timestamp").desc_nulls_last(),
        F.col("source_file").asc_nulls_last(),
        F.col("_source_row_hash").asc()
    ))
)

transfer_duplicate_rows = (
    transfer_ranked.filter(F.col("_row_num") > 1)
    .withColumn("rule_id", F.lit("TRF-05"))
    .withColumn("rejection_reason", F.lit("Duplicate transfer_id; deterministic survivor retained"))
    .withColumn("source_entity", F.lit("BED_TRANSFER_EVENTS"))
    .withColumn("rejected_at", F.current_timestamp())
)

silver_transfers = transfer_ranked.filter(F.col("_row_num") == 1).drop(
    "_row_num", "_encounter_found", "_from_bed_key", "_from_bed_found",
    "_to_bed_key", "_to_bed_found", "_source_row_hash",
    "rule_id", "rejection_reason", "source_entity", "rejected_at"
)

print("Bronze bed transfers:", transfers.count())
print("Silver bed transfers:", silver_transfers.count())
print("Rejected bed transfers:", transfer_rejected_basic.count())
print("Duplicate bed transfers:", transfer_duplicate_rows.count())

Bronze bed transfers: 392
Silver bed transfers: 392
Rejected bed transfers: 0
Duplicate bed transfers: 0


## 11. Build one traceable quarantine dataset

The project specification requires rejected rows with a rule ID, reason and source-file lineage. This table contains mandatory validation failures and duplicate extras. Optional missing encounter categories remain in Silver with a quality flag. The current ZIP has no separate status CDC file, so CDC history/current-state tables are not fabricated.

In [0]:
rejected_records = (
    rejected_patients
    .unionByName(rejected_departments, allowMissingColumns=True)
    .unionByName(rejected_beds, allowMissingColumns=True)
    .unionByName(encounter_rejected_basic, allowMissingColumns=True)
    .unionByName(encounter_duplicate_rows, allowMissingColumns=True)
    .unionByName(lab_rejected_basic, allowMissingColumns=True)
    .unionByName(lab_duplicate_rows, allowMissingColumns=True)
    .unionByName(transfer_rejected_basic, allowMissingColumns=True)
    .unionByName(transfer_duplicate_rows, allowMissingColumns=True)
    .withColumn("quarantined_at", F.current_timestamp())
)

(rejected_records.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_REJECTED_TABLE))

duplicate_audit = (
    duplicate_patients.withColumn("source_entity", F.lit("PATIENTS"))
    .unionByName(duplicate_departments.withColumn("source_entity", F.lit("DEPARTMENTS")), allowMissingColumns=True)
    .unionByName(duplicate_beds.withColumn("source_entity", F.lit("BEDS")), allowMissingColumns=True)
    .unionByName(encounter_duplicate_rows, allowMissingColumns=True)
    .unionByName(lab_duplicate_rows, allowMissingColumns=True)
    .unionByName(transfer_duplicate_rows, allowMissingColumns=True)
)

(duplicate_audit.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_DUPLICATE_AUDIT_TABLE))

print("Saved quarantine table:", SILVER_REJECTED_TABLE)
print("Quarantine rows:", rejected_records.count())
print("Saved duplicate audit:", SILVER_DUPLICATE_AUDIT_TABLE)

Saved quarantine table: capstone_8hospital.silver_schema.silver_rejected_records
Quarantine rows: 47
Saved duplicate audit: capstone_8hospital.silver_schema.silver_duplicate_audit


## 12. Write curated Silver tables

In [0]:
silver_outputs = {
    SILVER_PATIENT_TABLE: silver_patients,
    SILVER_DEPARTMENT_TABLE: silver_departments,
    SILVER_BED_TABLE: silver_beds,
    SILVER_ENCOUNTER_TABLE: silver_encounters,
    SILVER_LAB_TABLE: silver_labs,
    SILVER_TRANSFER_TABLE: silver_transfers
}

for table_name, dataframe in silver_outputs.items():
    (dataframe.write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(table_name))
    print("Saved:", table_name, "| Rows:", dataframe.count())

Saved: capstone_8hospital.silver_schema.silver_patients | Rows: 183
Saved: capstone_8hospital.silver_schema.silver_departments | Rows: 192
Saved: capstone_8hospital.silver_schema.silver_beds | Rows: 201
Saved: capstone_8hospital.silver_schema.silver_patient_encounters | Rows: 12329
Saved: capstone_8hospital.silver_schema.silver_lab_test_events | Rows: 758
Saved: capstone_8hospital.silver_schema.silver_bed_transfer_events | Rows: 392


## 13. Identifier protection in sample outputs

The Silver tables retain keys for authorized joins. Mask identifiers in notebook previews and apply Unity Catalog grants or approved masking policies before granting broad access.

In [0]:
display(
    spark.table(SILVER_ENCOUNTER_TABLE)
    .select(
        F.concat(F.lit("***"), F.substring("encounter_id", -4, 4)).alias("encounter_id_masked"),
        F.concat(F.lit("***"), F.substring("patient_id", -4, 4)).alias("patient_id_masked"),
        "department_id", "encounter_type", "encounter_status",
        "admission_ts", "discharge_ts", "length_of_stay_minutes", "dq_status"
    )
    .limit(20)
)

encounter_id_masked,patient_id_masked,department_id,encounter_type,encounter_status,admission_ts,discharge_ts,length_of_stay_minutes,dq_status
***0001,***0037,DEPART0800191,INPATIENT,DISCHARGED,2026-07-01T04:30:00.000Z,2026-07-01T04:35:00.000Z,5.0,VALID
***0002,***0093,DEPART0800136,INPATIENT,TRANSFERRED,2026-07-01T04:37:00.000Z,2026-07-01T04:42:00.000Z,5.0,VALID
***0003,***0040,DEPART0800127,EMERGENCY,ADMITTED,2026-07-01T04:44:00.000Z,2026-07-01T04:49:00.000Z,5.0,VALID
***0004,***0117,DEPART0800098,INPATIENT,ADMITTED,2026-07-01T04:51:00.000Z,2026-07-01T04:56:00.000Z,5.0,VALID
***0005,***0095,DEPART0800112,OUTPATIENT,TRANSFERRED,2026-07-01T04:58:00.000Z,2026-07-01T05:03:00.000Z,5.0,VALID
***0006,***0015,DEPART0800122,OUTPATIENT,DISCHARGED,2026-07-01T05:05:00.000Z,2026-07-01T05:10:00.000Z,5.0,VALID
***0007,***0123,DEPART0800037,OUTPATIENT,TRANSFERRED,2026-07-01T05:12:00.000Z,2026-07-01T05:17:00.000Z,5.0,VALID
***0008,***0124,DEPART0800112,OUTPATIENT,DISCHARGED,2026-07-01T05:19:00.000Z,2026-07-01T05:24:00.000Z,5.0,VALID
***0009,***0028,DEPART0800101,OUTPATIENT,DISCHARGED,2026-07-01T05:26:00.000Z,2026-07-01T05:31:00.000Z,5.0,VALID
***0010,***0169,DEPART0800141,EMERGENCY,TRANSFERRED,2026-07-01T05:33:00.000Z,2026-07-01T05:38:00.000Z,5.0,VALID


## 14. Unit tests for valid, invalid, duplicate and timestamp cases

In [0]:
from datetime import datetime

test_rows = [
    ("E-VALID", "P-1", "D-1", "2026-01-01 08:00:00", "2026-01-02 08:00:00"),
    ("", "P-1", "D-1", "2026-01-01 08:00:00", None),
    ("E-BAD-DATE", "P-1", "D-1", "not-a-date", None),
    ("E-BAD-STAY", "P-1", "D-1", "2026-01-02 08:00:00", "2026-01-01 08:00:00")
]

test_df = spark.createDataFrame(
    test_rows,
    ["encounter_id", "patient_id", "department_id", "admission_ts", "discharge_ts"]
).withColumn("admission_ts", F.try_to_timestamp("admission_ts")).withColumn(
    "discharge_ts", F.try_to_timestamp("discharge_ts")
)

assert test_df.filter(F.col("encounter_id") == "").count() == 1
assert test_df.filter(F.col("admission_ts").isNull()).count() == 1
assert test_df.filter(
    F.col("discharge_ts").isNotNull() &
    (F.col("discharge_ts") < F.col("admission_ts"))
).count() == 1

test_dups = spark.createDataFrame(
    [("E-DUP", 1), ("E-DUP", 2)],
    ["encounter_id", "source_sequence"]
)
test_window = Window.partitionBy("encounter_id").orderBy(F.col("source_sequence").desc())
assert test_dups.withColumn("_rn", F.row_number().over(test_window)).filter(F.col("_rn") == 1).count() == 1

print("PASS: missing encounter key detection")
print("PASS: invalid admission timestamp detection")
print("PASS: discharge-before-admission detection")
print("PASS: deterministic duplicate survivor")
print("NOTE: CDC late/out-of-order tests are not run because the current source ZIP has no CDC dataset.")

PASS: missing encounter key detection
PASS: invalid admission timestamp detection
PASS: discharge-before-admission detection
PASS: deterministic duplicate survivor
NOTE: CDC late/out-of-order tests are not run because the current source ZIP has no CDC dataset.


## 15. Final row-count and quality summary

In [0]:
summary_rows = [
    ("Bronze patient encounters", encounters.count()),
    ("Silver patient encounters", spark.table(SILVER_ENCOUNTER_TABLE).count()),
    ("Bronze lab test events", labs.count()),
    ("Silver lab test events", spark.table(SILVER_LAB_TABLE).count()),
    ("Bronze bed transfer events", transfers.count()),
    ("Silver bed transfer events", spark.table(SILVER_TRANSFER_TABLE).count()),
    ("Silver patients", spark.table(SILVER_PATIENT_TABLE).count()),
    ("Silver departments", spark.table(SILVER_DEPARTMENT_TABLE).count()),
    ("Silver beds", spark.table(SILVER_BED_TABLE).count()),
    ("Silver rejected records / quarantine", spark.table(SILVER_REJECTED_TABLE).count()),
    ("Silver duplicate audit", spark.table(SILVER_DUPLICATE_AUDIT_TABLE).count())
]

display(spark.createDataFrame(summary_rows, ["validation_check", "row_count"]))

print("Quarantine table schema:")
spark.table(SILVER_REJECTED_TABLE).printSchema()

display(
    spark.table(SILVER_REJECTED_TABLE)
    .groupBy("source_entity", "rule_id", "rejection_reason")
    .count()
    .orderBy("source_entity", "rule_id")
)

validation_check,row_count
Bronze patient encounters,12376
Silver patient encounters,12329
Bronze lab test events,758
Silver lab test events,758
Bronze bed transfer events,392
Silver bed transfer events,392
Silver patients,183
Silver departments,192
Silver beds,201
Silver rejected records / quarantine,47


Quarantine table schema:
root
 |-- patient_id: string (nullable = true)
 |-- synthetic_patient_code: string (nullable = true)
 |-- age_band: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- consent_flag: string (nullable = true)
 |-- insurance_type: string (nullable = true)
 |-- patient_status: string (nullable = true)
 |-- primary_language: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- rule_id: string (nullable = true)
 |-- rejection_reason: string (nullable = true)
 |-- source_entity: string (nullable = true)
 |-- rejected_at: timestamp (nullable = true)
 |-- _row_hash: string (nullable = true)
 |-- _row_num: integer (nullable = true)
 |-- department_id: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- specialty: string (nullable = true)
 |-- hospital_site: string (nullable = tru

source_entity,rule_id,rejection_reason,count
PATIENT_ENCOUNTERS,ENC-07,Duplicate encounter_id; deterministic survivor retained,47
